# Fraud Compliance Agent Notebook 08 — Fast-path model training and evaluation

**Fraud Compliance Agent · Post-Phase-0 candidate evaluation**  
**Status:** CRISP-DM runner over the tested `apps/api/modelling` library — synthetic demonstration by default  
**Decision supported:** Candidate model-release review only; never an autonomous payment decision

---

## In plain English

This notebook is the **practice ground for the quick first-pass model**. It shows how several candidate models would be trained and compared using an agreed dataset, while measuring useful trade-offs such as catching risky cases versus wrongly flagging legitimate ones.

The default run uses made-up data to demonstrate the mechanics safely. It is like a driving simulator: it lets us test the dashboard and evaluation process, but it is not evidence that the model can drive on real roads. Results here never make a payment decision or release a model into the app.

## Table of contents

1. [Business Understanding](#business-understanding)
2. [Data Understanding](#data-understanding)
3. [Data Preparation](#data-preparation)
4. [Modelling](#modelling)
5. [Evaluation](#evaluation)
6. [Deployment](#deployment)
7. [Richer-feature candidate (proposed v2)](#richer-features)
8. [References](#8-references)

### How to use this notebook

- **`synthetic`** is the default: it renders the full demonstration with generated fixtures. Every metric and chart is visibly labelled mechanics-only, never fraud-model evidence.
- **`gate`** is available through the FCA_NOTEBOOK08_MODE environment variable when you want to confirm that real training remains blocked.
- **`FCA_NOTEBOOK08_RICHER=run`** opts in to section 7, the proposed richer-feature experiment on the local Sparkov files (about six minutes). It is skipped by default, and the safe synthetic runner never starts it.
- **`approved`** is available only after an accepted, checksum-verified local corpus and model-training contract exist. It produces candidate evidence, never a runtime release.

### Where the logic lives

Every step below calls the `modelling` package in `apps/api`, so the behaviour is unit-tested rather than defined in cells:

| Module | Responsibility |
| --- | --- |
| `modelling.config` | Loads `config/fast-path-model-training.v1.json`: seed, hyperparameters, threshold grid, fixture shape |
| `modelling.paths` | Locates the repository root and the Git revision recorded in the report |
| `modelling.datasets` | Accepted-contract loading, checksum verification, the synthetic fixture, and schema validation |
| `modelling.features` | Chronological partition selection and feature-matrix assembly |
| `modelling.training` | The baseline and candidate pipelines and their held-out scores |
| `modelling.evaluation` | Aggregate metrics, the threshold sweep, slice metrics, and the reliability curve |
| `modelling.diagnostics` | The five Plotly figures |
| `modelling.report` | Run modes, the report body, its digest, and the approved-only write guard |
| `modelling.pipeline` | Run setup and the mode-dependent data-loading decision |
| `modelling.richer_features` | Point-in-time card, merchant and category features for section 7, built only from earlier rows |
| `modelling.richer_benchmark` | Section 7's config (`config/fast-path-model-richer-features.v2.candidate.json`), selection, calibration and model-fit checks |

Its tests are `apps/api/tests/test_modelling_*.py`; run them with `make api-test`. Changing a parameter means editing the versioned configuration file, not a cell.

This is a CRISP-DM notebook, not a production scoring service. The API remains the only eventual runtime home for approved, tested logic.




<a id="business-understanding"></a>
## 1. Business Understanding

### 1.1 Decision to support

The business question is: **does a proposed tabular model improve the fast-path risk estimate for an approved target, without compromising policy controls or customer outcomes?**

The model may estimate risk. It cannot approve, challenge, hold, release, or execute a payment. Deterministic fraud and APP controls, authority, oversight, and review remain independent and precede any action.

### 1.2 Success criteria and constraints

Success is not a single accuracy number. A candidate must be assessed against approved prevalence, PR-AUC, ROC-AUC, calibration, precision, recall, false-positive rate, cohort slices, operational review capacity, and fraud/false-decline cost assumptions.

The target, corpus, feature set, chronological partitions, calibration procedure, and release criteria must be accepted before real training. A model cannot stand in for an APP control if its target does not represent APP risk.

### 1.3 Run context and reproducibility

The next cell calls `prepare_run`, which selects the controlled execution mode, loads the versioned configuration, records the repository revision and seed, and resolves where this run may write. It opens no dataset and trains no model.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

from modelling.datasets import REQUIRED_ACCEPTED_FIELDS, load_accepted_manifest
from modelling.diagnostics import build_diagnostics
from modelling.evaluation import evaluate_models
from modelling.features import split_partitions
from modelling.paths import find_repository_root
from modelling.pipeline import load_dataset, mode_notice, prepare_run
from modelling.report import (
    APPROVED_MODE,
    GATE_MODE,
    evaluation_report,
    gated_report,
    report_location,
    write_report,
)
from modelling.training import train_models

# Paths resolve from the repository root, never from the notebook's working
# directory, which depends on the editor that opened it.
REPOSITORY_ROOT = find_repository_root(Path.cwd().resolve())
# prepare_run resolves the execution mode, loads the versioned configuration,
# and decides where this run may write. It opens no dataset and trains nothing.
setup = prepare_run(REPOSITORY_ROOT)

print("Mode:", setup.mode)
print("Configuration:", setup.config.config_version, "| seed:", setup.config.random_seed)
print("Revision:", setup.context["git_revision"])
print("Report destination:", report_location(setup.report_path, REPOSITORY_ROOT))
print("Safety: no raw data, identifiers, model weights, or release decision is written to Git.")

<a id="data-understanding"></a>
## 2. Data Understanding

### 2.1 Required evidence

Approved mode requires a versioned model-training contract that declares the local dataset path and checksum, target, feature columns, precomputed partitions, calibration procedure, slices, and release-criteria version. This prevents a notebook user from silently changing the question, labels, or data.

### 2.2 What we know today

The accepted contract currently authorises a Sparkov **mechanics-only** benchmark, not a production corpus or target. Plaid Sandbox observations remain integration evidence, not labelled fraud data. Approved mode consumes only the contract-declared, checksum-verified local dataset; it never fetches, manufactures, or silently substitutes data. Any results must remain labelled synthetic benchmark mechanics, not fraud-model performance evidence.

The next cell calls `load_accepted_manifest`, which enforces those boundaries before any dataset is considered. It refuses a contract that is absent, incomplete, or not explicitly accepted.



In [ ]:
print(mode_notice(setup.mode))

# Approved mode is the only path that reads an accepted contract. Loading it
# here fails the run early if it is absent, incomplete, or not yet accepted.
manifest = load_accepted_manifest(REPOSITORY_ROOT) if setup.mode == APPROVED_MODE else None
if manifest is not None:
    print("Training scope:", manifest["training_scope"])
    print("Only the contract's declared local dataset path and checksum are used.")
print("Contract fields required before any real training:")
print(", ".join(sorted(REQUIRED_ACCEPTED_FIELDS)))

<a id="data-preparation"></a>
## 3. Data Preparation

### 3.1 Prepare an analysis-ready, point-in-time dataset

This stage validates only the contract-declared schema. It checks the dataset checksum, required columns, target, and distinct train/calibration/test partitions. It does not repair missing fields, impute a target, resample classes, or invent unavailable signals.

### 3.2 Leakage and feature-parity guardrails

Real inputs must use the accepted point-in-time feature contract from Notebooks 04–07. A feature is eligible only if it can be computed from facts available before decision time and has an online-equivalent calculation. Chronological partitions are mandatory; a random split is not substituted here.

The next cell calls `load_dataset`, which loads either the seeded synthetic fixture or the accepted local CSV and then validates the declared columns and partitions.


In [ ]:
# load_dataset generates the seeded fixture in synthetic mode, reads the
# checksum-verified local CSV in approved mode, and reads nothing in gate mode.
# It validates the declared columns and partitions and repairs nothing.
dataset = load_dataset(setup)

if dataset is None:
    print("Gated: no dataset is loaded, so nothing downstream trains or evaluates.")
else:
    print("Dataset schema validated. Rows:", len(dataset.frame))
    print("Declared features:", ", ".join(dataset.contract.feature_columns))
    print("Declared partitions:", ", ".join(dataset.contract.partitions))
    print("Reporting slices:", ", ".join(dataset.contract.slice_columns))
    print("Input digest:", dataset.contract.dataset_sha256)

<a id="modelling"></a>
## 4. Modelling

### 4.1 Baseline and candidate

The class-balanced logistic-regression pipeline is the interpretable baseline. XGBoost is the proposed nonlinear tabular-model comparison, not an approved production algorithm. Both models use the same declared feature columns and the seed and hyperparameters recorded in `config/fast-path-model-training.v1.json`; they are a reproducible comparison only and must be reviewed under the accepted evaluation protocol. The candidate's class-imbalance weight is derived from the train partition at fit time, so it cannot drift from the data it describes.

### 4.2 Training protocol

Models fit only the train partition. The held-out test partition is never used to fit either model. The separate calibration partition is required by the contract for the approved calibration procedure; this notebook will not invent one.

The next cell calls `split_partitions` and `train_models`, which fit the baseline and XGBoost comparison pipelines and produce held-out scores only when synthetic or approved mode supplied data.



In [ ]:
if dataset is None:
    partitions = None
    model_scores = None
    print("Gated: model construction stays disabled until an accepted contract exists.")
else:
    # Fitting touches the train partition only. The calibration partition is
    # held back for the approved procedure, and the test partition is scored
    # once, after fitting.
    partitions = split_partitions(dataset.frame, dataset.contract)
    model_scores = train_models(partitions, setup.config)
    print("Partition rows:", partitions.counts)
    print("Target prevalence:", {name: round(rate, 6) for name, rate in partitions.prevalence.items()})
    print("Models fitted:", ", ".join(model_scores))
    print("No probability calibration, threshold, or promotion was applied.")

<a id="evaluation"></a>
## 5. Evaluation

### 5.1 Model quality versus business policy

PR-AUC, ROC-AUC, and Brier score describe score quality. A threshold sweep then shows the policy trade-off between precision, recall, false-positive rate, and block rate. No threshold is selected here: that is an independent policy, authority, and oversight decision.

### 5.2 Slice and limitation review

Aggregate performance can hide harm or operational regressions in a subgroup. The notebook therefore reports approved slice metrics where both classes are present. Synthetic results remain a software test only; they cannot establish fairness, fraud efficacy, or release readiness.

The next cell calls `evaluate_models`, which produces only aggregate diagnostics and never a payment action.


In [ ]:
if model_scores is None:
    models = None
    print("Gated: there is nothing to evaluate.")
else:
    # Aggregate metrics, the threshold sweep, and slice metrics are reported
    # together so score quality is never read as a policy decision.
    models = evaluate_models(partitions, model_scores, setup.config.evaluation)
    for name, values in models.items():
        print(name, {measure: round(value, 4) for measure, value in values["metrics"].items()})
    print("Threshold sweeps and slice metrics calculated; no operating threshold selected.")
    print("Metrics are", "synthetic mechanics only." if dataset.manifest is None else "candidate evidence pending review.")

### 5.3 Visual diagnostics

The next cell calls `build_diagnostics`, which returns the five figures; the notebook only decides whether to display them.

The following Plotly diagnostics use the Arbiris SDK notebook presentation convention: a white canvas, bold title, muted subtitle, explicit margins, readable hover details, and the established blue/red/green palette. In synthetic mode, every chart is visibly labelled as mechanics-only and cannot support a fraud-performance claim. In approved mode, it remains a candidate-evaluation view with no promotion or threshold decision.


In [ ]:
RENDER_PLOTS = os.getenv("FCA_NOTEBOOK08_RENDER_PLOTS", "true").strip().lower() not in {
    "0", "false", "no",
}

if models is None:
    figures = {}
    print("Visual diagnostics are gated until synthetic mechanics mode or an accepted contract supplies data.")
else:
    figures = build_diagnostics(partitions, model_scores, models, setup.config.evaluation, setup.mode)
    print("Created visual diagnostics:", ", ".join(figures))

# Rendering is optional so a headless or automated run produces no figure output.
if RENDER_PLOTS:
    for figure in figures.values():
        figure.show()
elif figures:
    print("Plot rendering disabled for this run; set FCA_NOTEBOOK08_RENDER_PLOTS=true to display figures.")

<a id="deployment"></a>
## 6. Deployment

### 6.1 Candidate-release evidence, not deployment

In CRISP-DM, deployment here means placing a reviewable result into the delivery process. The report records aggregate metrics, configuration identity, partition counts, limitations, and a digest of its own payload. It does not save model weights to Git, expose raw data, choose a policy threshold, or change API behaviour.

The next cell calls `evaluation_report` (or `gated_report`) and `write_report`. Only an approved-mode run resolves to the reviewed report in `docs/proposals/`; a gate or synthetic run writes a temporary file, and `FCA_NOTEBOOK08_REPORT_PATH` can redirect any run.

### 6.2 Required next steps before any runtime use

An independently approved release process must verify artifact storage/checksum, online–offline feature parity, deterministic-control precedence, contract version, monitoring and drift plan, rollback plan, and an authorised human decision. Only then may reusable implementation move into tested API code.


In [ ]:
report = (
    gated_report(setup.context)
    if setup.mode == GATE_MODE
    else evaluation_report(setup.mode, setup.context, dataset.contract, partitions, models)
)
# write_report adds the payload digest and writes canonical JSON. Only an
# approved-mode run resolves to the reviewed report under docs/proposals/;
# every other mode writes a temporary file that cannot be mistaken for it.
written = write_report(report, setup.report_path)

print("Sanitised report written:", report_location(setup.report_path, REPOSITORY_ROOT))
print("Report status:", written["status"])
print("Report SHA-256:", written["report_sha256"])
print("No model weights, source rows, identifiers, or policy threshold were written.")

<a id="richer-features"></a>
## 7. Richer-feature candidate (proposed v2)

### 7.1 Decision question, status and non-goals

**Decision question:** can richer point-in-time features, tuned only on rows the test partition never touches, lift the Sparkov benchmark, and do the model-fit checks show that the lift is genuine learning rather than leakage?

**Status:** proposed and offline. It is separate from the accepted v1 contract above: it does not amend v1, does not replace its reviewed report, and approves no feature for runtime.

**Non-goals:** no operating threshold, no model promotion, no runtime scoring, and no claim of production fraud performance. Sparkov fraud is produced by a simulator.

### 7.2 Inputs, data boundary and environment

- **Configuration:** `config/fast-path-model-richer-features.v2.candidate.json` holds the seed, feature groups, search grid, calibration method and fit checks. Change a parameter there, not in a cell.
- **Data:** the local, git-ignored Sparkov files. Only six source columns are read. Names, addresses, coordinates, date of birth and job never enter memory, and the card and merchant keys become anonymous codes used only to group earlier history.
- **Partitions:** identical to the accepted benchmark (train 1,037,340, calibration 259,335, test 555,719 rows), checked against those counts on every run. Model selection uses the first chronological half of the calibration partition, probability calibration uses the second half, and the test partition is scored once.
- **Environment:** the experiment makes 16 model fits, so it is opt-in. Set `FCA_NOTEBOOK08_RICHER=run`.
- **Expected output:** one sanitised, aggregate-only report with no rows, identifiers, model weights or thresholds. Its tests are `apps/api/tests/test_modelling_richer_features.py` and `test_modelling_richer_benchmark.py`.

In [ ]:
from modelling.paths import git_revision
from modelling.richer_benchmark import (
    load_reference_metrics,
    load_richer_config,
    richer_report_path,
    run_experiment,
)
from modelling.richer_features import load_raw_sparkov, raw_sparkov_paths

# The experiment makes 16 model fits (roughly six minutes), so it is
# opt-in and the safe synthetic runner never starts it.
train_path, test_path = raw_sparkov_paths(REPOSITORY_ROOT)
RUN_RICHER = os.getenv("FCA_NOTEBOOK08_RICHER", "skip").strip().lower() == "run"
if not (RUN_RICHER and train_path.is_file() and test_path.is_file()):
    richer = None
    print("Skipped. Set FCA_NOTEBOOK08_RICHER=run with the local Sparkov files present.")
else:
    richer = run_experiment(
        load_raw_sparkov(train_path, test_path),
        load_richer_config(REPOSITORY_ROOT),
        load_reference_metrics(REPOSITORY_ROOT),
        git_revision(REPOSITORY_ROOT),
    )

### 7.3 Inspect the fit checks and record the report

The next cell prints the headline held-out result, the shuffled-label control and the feature-group ablation, then writes the aggregate-only proposed report. It does nothing when the experiment was skipped.

In [ ]:
if richer is not None:
    checks = richer["fit_checks"]
    print("Held-out test:", {k: round(v, 4) for k, v in checks["partition_metrics"]["test"].items() if k in {"pr_auc", "roc_auc", "prevalence"}})
    print("Shuffled-label control:", {k: round(v, 4) for k, v in checks["shuffled_label_control"].items()})
    for name, values in checks["ablation"]["cumulative"].items():
        print(f"{name}: test PR-AUC {values['test_pr_auc']:.3f}")
    # Aggregate-only and destined for review; it is not the accepted v1 report.
    written = write_report(richer, richer_report_path(REPOSITORY_ROOT))
    print("Proposed report SHA-256:", written["report_sha256"])

### 7.4 Findings, limitations and recommendation

**Recorded run** (report `docs/proposals/fast-path-model-richer-features.proposed.json`; the numbers below describe that run and are not recomputed when the notebook is read):

- **Lift.** Held-out test PR-AUC is 0.961 and ROC-AUC 0.9992, against 0.432 and 0.979 for the accepted four-feature XGBoost (test prevalence 0.0039). Recall is 94.5% at a 0.1% false-positive rate and 98.9% at 1%.
- **Where it comes from.** Tuning alone barely moves the four-feature model (0.437). The lift comes from features: merchant category takes PR-AUC to 0.785 and card velocity and timing to 0.963. The familiarity group adds nothing measurable (the other 30 features score 0.963), and amount-versus-history is largely redundant once velocity is present (0.955 without it).
- **Fit.** A shuffled-label control scores exactly the base rate (PR-AUC 0.0039), so the pipeline is not leaking through its structure. Train PR-AUC is 1.000 against 0.981 on the selection rows and 0.961 on test: mild overfitting plus later-period drift, with a learning curve that is flat after about 330 trees. Monthly test PR-AUC stays between 0.94 and 0.97.
- **Calibration.** Isotonic calibration changes the Brier score only from 0.00055 to 0.00054 and worsens log loss, so it is not a demonstrated improvement. The top score bin over-predicts (about 0.71 predicted against 0.68 observed), and the fraud rate falls from 0.58% in train to 0.39% in test.

**Limitations and unknowns.** This is a simulator: category and amount carry much of the signal, so the score shows the model learned the generator's patterns, not real fraud performance. The card velocity features need online streaming aggregates whose parity is not established. There is a single seed and 2,145 test positives. Controls can rule out leakage through the pipeline but not simulator artefacts.

**Recommendation:** proposed. Keep this as benchmark evidence only. Do not promote a model or select a threshold. Before any v2 contract is accepted, confirm which features have an online equivalent and decide how a threshold and recalibration would be approved.

<a id="8-references"></a>
## 8. References

### Fraud-ML and evaluation

1. Stripe. (2021, December 15). *A primer on machine learning for fraud detection*. Stripe Radar Technical Guide. https://stripe.com/gb/guides/primer-on-machine-learning-for-fraud-protection  
   Used for the separation of model quality from policy thresholds, precision/recall and false-positive trade-offs, held-out evaluation, real-time feature parity, monitoring, and drift.
2. scikit-learn developers. (n.d.). *Model evaluation: quantifying the quality of predictions*. scikit-learn documentation. https://scikit-learn.org/stable/modules/model_evaluation.html  
   Used as the implementation reference for the aggregate evaluation metrics in this mechanics harness.

### Code-style convention

3. van delay. (2024). *Does it really matter what order you import modules in Python?* r/Python discussion. https://www.reddit.com/r/Python/comments/1bv9of6/does_it_really_matter_what_order_you_import/  
   Applied as a convention: standard-library imports, then third-party imports, then project-local imports, with blank lines between groups. Ruff's isort-compatible `I` rules enforce this convention.
4. Astral. (n.d.). *Ruff rule reference: isort (I)*. https://docs.astral.sh/ruff/rules/  
   Project enforcement reference for import ordering.

No other research paper or external fraud-data blog was used to create this notebook. The synthetic fixture exists only to test mechanics and is not a research source.


## Review checklist

- [ ] Confirm the execution mode was appropriate and visible in the report.
- [ ] Confirm approved mode used an accepted contract and checksum-verified local data.
- [ ] Confirm synthetic output is labelled mechanics-only.
- [ ] Review model metrics, calibration procedure, slices, and threshold trade-offs together.
- [ ] Confirm no threshold, release, or payment authority was inferred from this notebook.
- [ ] Confirm the `modelling` library's tests pass (`make api-test`) and that any parameter change was made in `config/fast-path-model-training.v1.json`.
- [ ] Clear notebook outputs before commit.
- [ ] Recommendation remains **proposed**: no runtime promotion or policy decision is implied.
